In [ ]:
import polars as pl
from datetime import datetime, timedelta
from import_data import rename_cols, convert_cols_to_numeric, get_full_connection_times, get_times_connections
import time
import numpy as np
from scipy.sparse import csr_array
from scipy.sparse.csgraph import minimum_spanning_tree

In [ ]:
def find_next_branch(
    services: pl.DataFrame,
    start_station: str,
    start_time: int,

    ) -> pl.DataFrame:
    """
    Find the possible reachable stations from current station, and the arrival times

    Args:
        services: df of services
        start_station: String of station to look from
        start_time: int of time to look from (ie departure time from specified station)
    
    Returns:
        possible_stations: df of station crs and arrival times from start station
    """
    # Find the services we can take from the start station
    services_from_start = services.filter((pl.col("crs") == start_station) & (pl.col("departure") > start_time)).select(("departure", "serviceUid")).rename({"departure": "time"})
    # Now find where these services can take us
    possible_stations = services.join(services_from_start, on = "serviceUid", how="inner").filter(pl.col("arrival") > pl.col("time")).drop("time").sort("arrival").group_by("crs").first().select("crs", "arrival")
    return possible_stations

In [ ]:
def test_branch(
        branches: dict[str, dict[str, int | list[dict]]],
        stations: list[str],

    ) -> bool:
    """
    Check if a branch has reached all stations
    Creates sub function iter_branches to test each branch recursively

    Args:
        branches: Dict of connections
        stations: List of stations to reach

    Returns:
        Bool: True if has reached all stations in 1 path, False if not
    """
    stations_original = stations.copy()
    def iter_branch(branches: dict[str, dict[str, int | list[dict]]], stations: list[str]):
        while len(branches) != 0:
            current_station = list(branches.keys())[0]
            if current_station in stations:
                stations.remove(current_station)
                if stations == []:
                    return True
            print(f"{current_station=}, {stations=}")
            for next_station in branches[current_station]["reachable"]:
                print(f"Testing with {list(next_station.keys())[0]=}, {stations=}")
                result = iter_branch(next_station, stations)
                if result:
                    return True
            print(f"{current_station}: {current_station in stations_original}, {stations_original}")
            if current_station in stations_original and current_station not in stations:
                stations.append(current_station)
            return False
    if iter_branch(branches, stations):
        return True
    return False

In [ ]:
# branches = {"EXD": {"reachable": [{"EXC": {"reachable": [{"EXD": {"reachable": [{"PLY": {"reachable": []}}]}}, {"EXM": {"reachable": []}}]}}, {"PLY": {"reachable": [{"EXD": {"reachable": [{"EXC": {"reachable": []}}]}}, {"PNZ": {"reachable": []}}]}}]}}
branches = {"EXD": {"reachable": [{"EXC": {"reachable": [{"EXM": {"reachable": []}}]}}, {"PLY": {"reachable": [{"EXD": {"reachable": [{"EXC": {"reachable": []}}]}}, {"PNZ": {"reachable": []}}]}}]}}

In [ ]:
test_branch(branches, ["EXD", "PLY", "EXC"])

In [ ]:
def branching_graph(
        services: pl.DataFrame,
        stations: list[str],
        start_time: datetime,
        start_station: str = "",
        change_time: int = 5

    ) -> list[tuple[datetime, list[str]]]:
    """
    
    """
    if start_station != "":
        start_stations: list[str] = [start_station]
    else:
        start_stations: list[str] = stations

    start_time_num: int = int(start_time.strftime("%H%M"))

    branches: dict[str, dict[str, int | list[dict]]] = {} # Recursive in nature, so the final dict in type description contains the same form as the type descrtiption

    for start_station in start_stations:
        possible_stations = find_next_branch(services, start_station, start_time_num)
        branches[start_station] = {"time": start_time_num, "reachable": [{crs: {"time": arrival} for crs, arrival in possible_stations.iter_rows()}]}


In [ ]:
start_time = 900
services = convert_cols_to_numeric(pl.read_csv("Services.csv", infer_schema=None))
stations = ["EXD", "IPS", "EXC", "PLY"]

In [ ]:
services

In [ ]:
start_station = "EXD"

In [ ]:
find_next_branch(services, "EXD", start_time=start_time)

In [ ]:
services_from_start = services.filter((pl.col("crs") == start_station) & (pl.col("departure") > start_time)).select(("departure", "serviceUid")).rename({"departure": "time"})
possible_stations = services.join(services_from_start, on = "serviceUid", how="inner").filter(pl.col("arrival") > pl.col("time")).drop("time").sort("arrival").group_by("crs").first().select("crs", "arrival")

In [ ]:
{crs: {"time": arrival} for crs, arrival in possible_stations.iter_rows()}

In [ ]:
dict(zip(possible_stations["crs"], possible_stations["arrival"]))

In [ ]:
possible_stations